# Notebook 1 — AIS Input, Cleaning, Spatial Validation, and Visual Inspection

Stage ini mempertahankan kontrak ilmiah baseline: validasi skema, parsing waktu, prioritas alasan penolakan, penyaringan koridor, duplikasi, segmentasi lintasan, peta audit, dan keluaran CSV. Logika transformasi berada pada `src/mfar_stage12.py` agar dapat diuji tanpa mengubah hasil.


In [ ]:
#@title Inisialisasi repository dan jalur MFAR
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/yanto-mashardi/MFAR_Modular_Colab_Pipeline.git"
DEFAULT_REF = "refactor/prompt1-stage01-02"
REPOSITORY_REF = os.environ.get("MFAR_GIT_REF", DEFAULT_REF)

def locate_repository() -> Path:
    candidates = []
    override = os.environ.get("MFAR_CODE_ROOT", "").strip()
    if override:
        candidates.append(Path(override))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/MFAR_Modular_Colab_Pipeline"),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
    ])
    for candidate in candidates:
        if (candidate / "src" / "mfar_paths.py").is_file():
            return candidate.resolve()

    clone_target = Path("/content/MFAR_Modular_Colab_Pipeline")
    if "google.colab" in sys.modules or Path("/content").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
             REPOSITORY_URL, str(clone_target)],
            check=True,
        )
        if (clone_target / "src" / "mfar_paths.py").is_file():
            return clone_target.resolve()
    raise FileNotFoundError(
        "Repository MFAR tidak ditemukan. Tetapkan MFAR_CODE_ROOT atau clone repository terlebih dahulu."
    )

CODE_ROOT = locate_repository()
os.environ["MFAR_CODE_ROOT"] = str(CODE_ROOT)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, CONFIG_DIR,
    STAGE_01_DIR, STAGE_02_DIR,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)
print("Repository:", CODE_ROOT)
print("Input/output root:", STAGE_01_DIR.parent.parent)


In [ ]:
#@title Jalankan Stage 01: input, cleaning, segmentasi, audit, dan ekspor
from src.mfar_stage12 import run_stage1

NOTEBOOK_NAME = "01_AIS_Input_and_Cleaning.ipynb"
result = run_stage1(
    ais_path=AIS_RAW_PATH,
    vehicle_arrival_path=VEHICLE_ARRIVAL_PATH,
    config_dir=CONFIG_DIR,
    stage_dir=STAGE_01_DIR,
    notebook_name=NOTEBOOK_NAME,
    started_at=_MFAR_STARTED_AT,
)

print("AIS mentah:", len(result.raw))
print("AIS diterima:", len(result.clean))
print("AIS ditolak:", len(result.rejected))
print("Segmen valid:", len(result.segment_summary))


In [ ]:
#@title Laporan penyelesaian Stage 01
print("Stage 01 selesai.")
for path in result.saved_files:
    print("-", path.name)
